# Squadup — AI Hackathon Team Matching

**Run the cells in order. The last cell gives you a live public URL.**

| Cell | What it does |
|------|-------------|
| 1 | Install dependencies |
| 2 | Write backend + frontend files to disk |
| 3 | Start Flask server + expose via Colab proxy or ngrok |

In [ ]:
# ── Cell 1: Install deps ──
!pip install -q flask flask-cors scikit-learn numpy
print('✅ Dependencies ready')

In [ ]:
# ── Cell 2: Write backend + frontend to disk ──
# The full source is embedded here so the notebook is self-contained.
import base64, os

# — Backend —
BACKEND_SRC = '''
import json, os, datetime
from collections import Counter
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from flask import Flask, request, jsonify, Response
from flask_cors import CORS

USERS_DB = {}
TEAMS_DB = {}

SEED_USERS = [
    {"id":"arjun_iitm","username":"arjun.dev","name":"Arjun Sharma","email":"arjun@demo.com","college":"IIT Madras","education":"4th Year","github":"https://github.com/arjunsharma","linkedin":"https://linkedin.com/in/arjunsharma","xp":3420,"domains":["ML / AI","Full-stack","Data Engineering"],"badges":["SIH Winner","5+ Hackathons"],"connects":["priya_nit","karthik_bits"],"teams":[],"date_joined":"2022-08-01","reviews":[]},
    {"id":"priya_nit","username":"priya.design","name":"Priya Nair","email":"priya@demo.com","college":"NIT Trichy","education":"3rd Year","github":"https://github.com/priyanair","linkedin":"https://linkedin.com/in/priyanair","xp":2180,"domains":["Design / UX","Full-stack"],"badges":["Hack the Mountain"],"connects":["arjun_iitm"],"teams":[],"date_joined":"2023-01-15","reviews":[]},
    {"id":"karthik_bits","username":"karthik.hw","name":"Karthik Raj","email":"karthik@demo.com","college":"BITS Pilani","education":"4th Year","github":"https://github.com/karthikraj","linkedin":"","xp":4100,"domains":["Hardware / IoT","DevOps / Cloud"],"badges":["SIH Winner","Open Source Contributor"],"connects":["arjun_iitm"],"teams":[],"date_joined":"2021-06-10","reviews":[]},
    {"id":"sneha_iiith","username":"sneha.nlp","name":"Sneha Iyer","email":"sneha@demo.com","college":"IIIT Hyderabad","education":"Postgraduate","github":"https://github.com/sneha","linkedin":"https://linkedin.com/in/sneha","xp":2890,"domains":["NLP / LLMs","ML / AI"],"badges":["AI Hack Finalist"],"connects":[],"teams":[],"date_joined":"2022-03-20","reviews":[]},
    {"id":"dev_vit","username":"dev.cloud","name":"Dev Patel","email":"dev@demo.com","college":"VIT Vellore","education":"3rd Year","github":"https://github.com/devpatel","linkedin":"","xp":1650,"domains":["DevOps / Cloud","Full-stack"],"badges":[],"connects":[],"teams":[],"date_joined":"2023-07-01","reviews":[]},
    {"id":"riya_nsut","username":"riya.web3","name":"Riya Mehta","email":"riya@demo.com","college":"NSUT Delhi","education":"3rd Year","github":"https://github.com/riyamehta","linkedin":"https://linkedin.com/in/riyamehta","xp":2560,"domains":["Blockchain","Full-stack"],"badges":["ETHIndia Finalist"],"connects":[],"teams":[],"date_joined":"2022-11-05","reviews":[]},
    {"id":"aarav_dtu","username":"aarav.mobile","name":"Aarav Singh","email":"aarav@demo.com","college":"DTU Delhi","education":"4th Year","github":"https://github.com/aaravsingh","linkedin":"https://linkedin.com/in/aaravsingh","xp":5200,"domains":["Mobile","Full-stack"],"badges":["Google DevFest Top 3","HackCBS Winner","5+ Hackathons"],"connects":[],"teams":[],"date_joined":"2021-01-20","reviews":[]},
]
for u in SEED_USERS: USERS_DB[u["id"]] = u

TEAMS_DB["team_demo1"] = {"id":"team_demo1","name":"Team Firewall","hackathon":"Smart India Hackathon 2024","start_date":"2024-01-24","end_date":"2024-01-26","members":["arjun_iitm","karthik_bits","priya_nit"],"reviews":{"arjun_iitm":[{"from":"karthik_bits","from_name":"Karthik Raj","stars":5,"text":"Brilliant ML work, led the team perfectly"}],"karthik_bits":[{"from":"arjun_iitm","from_name":"Arjun Sharma","stars":5,"text":"Best hardware guy I have worked with"}],"priya_nit":[{"from":"arjun_iitm","from_name":"Arjun Sharma","stars":4,"text":"Great UI work under pressure"}]},"created_at":"2024-01-20"}
TEAMS_DB["team_demo2"] = {"id":"team_demo2","name":"Team Kernel","hackathon":"HackCBS 6.0","start_date":"2023-10-14","end_date":"2023-10-15","members":["sneha_iiith","dev_vit","riya_nsut"],"reviews":{},"created_at":"2023-10-10"}

SKILL_CLUSTERS = {
    "ml_ai":["python","tensorflow","pytorch","keras","mlops","nlp","computer vision","transformers","llm"],
    "frontend":["react","vue","typescript","javascript","css","figma","tailwind","next.js"],
    "backend":["node.js","fastapi","django","flask","go","rust","java","express","mongodb","postgresql"],
    "devops":["docker","kubernetes","aws","gcp","azure","terraform","linux","bash"],
    "mobile":["react native","flutter","swift","kotlin","ios","android","firebase"],
    "hardware":["fpga","verilog","arduino","pcb design","embedded c","c++","rtos","raspberry pi"],
    "blockchain":["solidity","web3","ethereum","smart contracts","polygon"],
    "data":["pandas","numpy","sql","spark","kafka","airflow","dbt","tableau"],
    "design":["figma","adobe xd","sketch","ui/ux","prototyping","design systems"],
    "cybersec":["penetration testing","burp suite","ctf","cryptography","kali linux","owasp"],
}

DOMAIN_TO_CLUSTER = {
    "ML / AI":"ml_ai","Full-stack":"frontend","Mobile":"mobile",
    "Hardware / IoT":"hardware","Data Engineering":"data","DevOps / Cloud":"devops",
    "Design / UX":"design","Blockchain":"blockchain","Cybersecurity":"cybersec","NLP / LLMs":"ml_ai",
}

class SquadupEngine:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(analyzer="word",lowercase=True,token_pattern=r"[a-zA-Z0-9][a-zA-Z0-9/\\.\\+\\-\\_# ]*",max_features=500,ngram_range=(1,2))
        self.fitted = False
        self._cache = {}
    def _to_text(self,u):
        parts=[]
        for d in u.get("domains",[]):
            c=DOMAIN_TO_CLUSTER.get(d,"")
            if c and c in SKILL_CLUSTERS: parts+=SKILL_CLUSTERS[c][:5]
            parts+=[d.lower().replace("/"," ").replace("-"," ")]*3
        for b in u.get("badges",[]): parts.append(b.lower())
        if u.get("college"): parts.append(u["college"].lower())
        return " ".join(parts) if parts else "hacker developer"
    def fit(self,users):
        texts=[self._to_text(u) for u in users]
        if not texts: return
        self.vectorizer.fit(texts); self.fitted=True; self._cache={}
        for u in users:
            uid=u.get("id")
            if uid: self._cache[uid]=self._vec(u)
    def _vec(self,u):
        if not self.fitted: return np.zeros(1)
        return self.vectorizer.transform([self._to_text(u)]).toarray()[0]
    def similarity(self,u1,u2):
        if not self.fitted: return 0.5
        v1=self._cache.get(u1.get("id")) or self._vec(u1)
        v2=self._cache.get(u2.get("id")) or self._vec(u2)
        raw=float(cosine_similarity([v1],[v2])[0][0])
        d1=set(u1.get("domains",[])); d2=set(u2.get("domains",[]))
        boost=len(d1&d2)*0.08-len((d1|d2)-(d1&d2))*0.03
        xp_score=max(0,1-abs(u1.get("xp",0)-u2.get("xp",0))/5000)
        return min(1.0,max(0.0,raw*0.5+max(0,boost)*0.3+xp_score*0.2))
    def pairwise_compat(self,u1,u2):
        v1=self._cache.get(u1.get("id")) or self._vec(u1)
        v2=self._cache.get(u2.get("id")) or self._vec(u2)
        raw=float(cosine_similarity([v1],[v2])[0][0]) if self.fitted else 0.5
        comp=max(0.0,min(1.0,1.0-abs(raw-0.25)*2.5))
        d1=set(u1.get("domains",[])); d2=set(u2.get("domains",[]))
        dom=min(0.3,len(d1&d2)*0.12)
        lv=max(0,1-abs(u1.get("xp",0)-u2.get("xp",0))/5000)
        return min(100,max(40,int(40+(comp*0.55+dom*0.25+lv*0.20)*60)))
    def team_compat(self,members):
        n=len(members)
        if n<2: return {"score":0,"message":"Need >=2","gaps":[],"strengths":[],"coverage_pct":0,"recommendation":""}
        all_d=[]
        for m in members: all_d.extend(m.get("domains",[]))
        covered=set()
        for d in all_d:
            c=DOMAIN_TO_CLUSTER.get(d)
            if c: covered.add(c)
        gaps=set(SKILL_CLUSTERS.keys())-covered
        pairs=[]; total=0
        for i in range(n):
            for j in range(i+1,n):
                s=self.pairwise_compat(members[i],members[j]); pairs.append(s); total+=s
        avg=total/len(pairs) if pairs else 50
        cov=len(covered)/len(SKILL_CLUSTERS)
        final=min(100,int(avg*0.75+cov*15+{2:0,3:8,4:10}.get(n,5)))
        rec=("Great coverage!" if not gaps else f"Consider adding {list(gaps)[0].replace(chr(95),' ').title()} skills.")
        return {"score":final,"gaps":[g.replace("_"," ").title() for g in gaps],"strengths":[c.replace("_"," ").title() for c in covered],"coverage_pct":int(cov*100),"recommendation":rec}
    def compose_team(self,anchor,pool,size=4):
        team=[anchor]; remaining=[p for p in pool if p.get("id")!=anchor.get("id")]
        while len(team)<size and remaining:
            best,best_s=None,-1
            for c in remaining:
                s=self.team_compat(team+[c])["score"]
                if s>best_s: best_s=s; best=c
            if best: team.append(best); remaining.remove(best)
            else: break
        return team
    def suggestions(self,user,pool,n=8):
        others=[u for u in pool if u.get("id")!=user.get("id")]
        scored=[{**u,"similarity_score":int(self.similarity(user,u)*100)} for u in others]
        return sorted(scored,key=lambda x:x["similarity_score"],reverse=True)[:n]

engine=SquadupEngine()
engine.fit(list(USERS_DB.values()))

def refit(): engine.fit(list(USERS_DB.values()))

app=Flask(__name__)
CORS(app,origins=["*"])
def ok(d): return jsonify({"success":True,"data":d})
def err(m): return jsonify({"success":False,"error":m})

FRONTEND_PATH=os.path.join(os.path.dirname(os.path.abspath(__file__)),"squadup_frontend.html")

@app.route("/")
def index(): return Response(open(FRONTEND_PATH,encoding="utf-8").read(),mimetype="text/html") if os.path.exists(FRONTEND_PATH) else "frontend not found"

@app.route("/api/health")
def health(): return ok({"status":"ok","users":len(USERS_DB),"teams":len(TEAMS_DB)})

@app.route("/api/users",methods=["GET","POST"])
def users_route():
    if request.method=="POST":
        p=request.json or {}
        if not p.get("id"): return err("Missing id")
        USERS_DB[p["id"]]=p; refit(); return ok(p)
    q=(request.args.get("q","")).lower(); domain=request.args.get("domain","")
    users=list(USERS_DB.values())
    if q: users=[u for u in users if q in (u.get("username","")+u.get("name","")+u.get("college","")).lower() or any(q in d.lower() for d in u.get("domains",[]))]
    if domain: users=[u for u in users if domain in u.get("domains",[])]
    return ok(users)

@app.route("/api/users/<uid>",methods=["GET","PATCH","DELETE"])
def user_route(uid):
    if request.method=="GET":
        u=USERS_DB.get(uid); return ok(u) if u else err("Not found")
    if request.method=="PATCH":
        if uid not in USERS_DB: return err("Not found")
        USERS_DB[uid].update(request.json or {}); refit(); return ok(USERS_DB[uid])
    USERS_DB.pop(uid,None); return ok({"deleted":uid})

@app.route("/api/users/<uid>/suggestions")
def user_suggestions(uid):
    u=USERS_DB.get(uid)
    return ok(engine.suggestions(u,list(USERS_DB.values()))) if u else err("Not found")

@app.route("/api/teams/compose",methods=["POST"])
def compose():
    body=request.json or {}; anchor=USERS_DB.get(body.get("anchor"))
    if not anchor: return err("Anchor not found")
    pool=list(USERS_DB.values()); size=min(int(body.get("size",4)),len(pool))
    team=engine.compose_team(anchor,pool,size)
    return ok({"team":team,"compatibility":engine.team_compat(team)})

@app.route("/api/teams",methods=["GET","POST"])
def teams_route():
    if request.method=="POST":
        t=request.json or {}
        if not t.get("id"): return err("Missing id")
        TEAMS_DB[t["id"]]=t; return ok(t)
    uid=request.args.get("user","")
    teams=list(TEAMS_DB.values())
    if uid: teams=[t for t in teams if uid in t.get("members",[])]
    return ok(teams)

@app.route("/api/teams/<tid>",methods=["GET","PATCH"])
def team_route(tid):
    t=TEAMS_DB.get(tid)
    if not t: return err("Not found")
    if request.method=="PATCH": TEAMS_DB[tid].update(request.json or {}); return ok(TEAMS_DB[tid])
    return ok(t)

@app.route("/api/teams/<tid>/reviews",methods=["POST"])
def add_review(tid):
    t=TEAMS_DB.get(tid)
    if not t: return err("Not found")
    body=request.json or {}; to=body.get("to")
    if not to: return err("Missing to")
    rv={"from":body.get("from",""),"from_name":body.get("from_name",""),"stars":int(body.get("stars",5)),"text":body.get("text",""),"date":datetime.date.today().isoformat()}
    t.setdefault("reviews",{}).setdefault(to,[]).append(rv)
    target=USERS_DB.get(to)
    if target: target["xp"]=target.get("xp",0)+{5:200,4:120,3:60}.get(rv["stars"],30)
    return ok(rv)

@app.route("/api/leaderboard")
def leaderboard():
    pool=sorted(list(USERS_DB.values()),key=lambda u:u.get("xp",0),reverse=True)
    for i,u in enumerate(pool): u["rank"]=i+1
    return ok(pool[:int(request.args.get("limit",20))])
'''

with open('squadup_app.py', 'w') as f:
    f.write(BACKEND_SRC.strip())
print('Backend written: squadup_app.py')

In [ ]:
# ── Cell 3: Write frontend ──
# The frontend HTML is stored as base64 to avoid any escaping issues
import base64, urllib.request

# Download from the file we generated
# If running locally with the files in the same folder, just copy them.
# In Colab, paste the frontend HTML file content as base64 below:
# (this cell expects squadup_frontend.html to already exist, e.g. uploaded)

import os
if os.path.exists('squadup_frontend.html'):
    print('Frontend file already present.')
else:
    # Fallback: create minimal placeholder
    with open('squadup_frontend.html','w') as f:
        f.write('<h1>Upload squadup_frontend.html to Colab files panel</h1>')
    print('Placeholder written. Upload the real squadup_frontend.html via Colab files panel.')
    print('Then re-run Cell 4.')

In [ ]:
# ── Cell 4: Start the server ──
# Method A: Colab built-in proxy (no token needed, always works)
# Method B: ngrok (public URL, optional)

import subprocess, sys, threading, time, os

# Kill anything on port 5050
os.system('fuser -k 5050/tcp 2>/dev/null || true')
time.sleep(1)

# Start Flask in background thread
import importlib.util, sys as _sys

# Load squadup_app as a module
spec = importlib.util.spec_from_file_location('squadup_app', 'squadup_app.py')
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

flask_app = mod.app

def run_flask():
    flask_app.run(host='0.0.0.0', port=5050, use_reloader=False, debug=False)

t = threading.Thread(target=run_flask, daemon=True)
t.start()
time.sleep(2)

# ── Method A: Colab built-in proxy (recommended, no setup needed) ──
try:
    from google.colab.output import eval_js
    url = eval_js('google.colab.kernel.proxyPort(5050)')
    print('=' * 64)
    print('  SQUADUP IS LIVE (Colab proxy)')
    print('=' * 64)
    print(f'  Website  ->  {url}')
    print(f'  API      ->  {url}/api/health')
    print()
    print('  Open the Website URL in your browser.')
    print('  NOTE: This URL works while your Colab session is active.')
    print('  For a persistent public URL, run Cell 5 (ngrok).')
    print('=' * 64)
except Exception:
    print('Running outside Colab. Visit http://localhost:5050')

In [ ]:
# ── Cell 5 (Optional): ngrok public URL ──
# Use this if you want to share the URL with teammates.
# Get a free token at https://dashboard.ngrok.com/auth

NGROK_TOKEN = 'PASTE_YOUR_NEW_TOKEN_HERE'  # <- replace this

if NGROK_TOKEN == 'PASTE_YOUR_NEW_TOKEN_HERE':
    print('Add your ngrok token above and re-run this cell.')
    print('Get one free at https://dashboard.ngrok.com/auth')
else:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'], check=True)
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    try: ngrok.kill()
    except: pass
    tunnel = ngrok.connect(5050)
    url = tunnel.public_url
    print('=' * 64)
    print('  SQUADUP IS LIVE (public ngrok URL)')
    print('=' * 64)
    print(f'  Website  ->  {url}')
    print(f'  API      ->  {url}/api/health')
    print()
    print('  Share this URL with teammates to let them register.')
    print('  Profiles are saved in memory for the session.')
    print('=' * 64)